# TaqSim: блок за блоком

*Семь типов блоков, одна небольшая река и карточка правил для каждого блока*

Д-р Тобиас Зигфрид, hydrosolutions GmbH · CC BY 4.0

<p><img src="https://hydrosolutions.github.io/zarafshan-taqsim-course/images/hydrosolutions_logo.png" alt="hydrosolutions" width="220"></p>
<p><b>hydrosolutions GmbH</b> · Venusstrasse 29, 8050 Zürich, Switzerland · <a href="https://www.hydrosolutions.ch">www.hydrosolutions.ch</a> · hs@hydrosolutions.ch · +41 43 535 05 80</p>

In [ ]:
# @title Выполнить расчёт / показать код
# ── Setup: run this cell first. In Colab it takes about a minute. ──────────────────────────────────
# It fetches the course folder, installs TaqSim, and tells Python where the course code lives.
import os
import sys
import subprocess

if not os.path.exists("src/zarafshan_taqsim"):  # not inside the course folder yet (the normal case in Colab)
    if not os.path.exists("zarafshan-taqsim-course"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/hydrosolutions/zarafshan-taqsim-course.git"], check=True)
    os.chdir("zarafshan-taqsim-course")
# A runtime that ran this cell earlier may hold an older copy of the course: bring the folder up to date and
# forget any course module Python has already loaded, so that the current code is what gets imported.
if os.path.exists(".git"):
    subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
for name in [m for m in sys.modules if m == "zarafshan_taqsim" or m.startswith("zarafshan_taqsim.")]:
    del sys.modules[name]
try:
    import taqsim  # noqa: F401  (already installed when you work on your own computer)
except ImportError:
    %pip install -q deap ctrl-freak pymoo
    %pip install -q --no-deps "git+https://github.com/hydrosolutions/taqsim.git@v0.1.4"
sys.path[:0] = [os.path.abspath("src")]
print("Ready. Working folder:", os.getcwd())


In [ ]:
# @title Выполнить расчёт / показать код
#| label: fig-catalogue
#| fig-cap: "The seven kinds of block. Every TaqSim model, including the 61-block Zarafshan model of this course, is made of these and nothing else."
#| echo: false
from zarafshan_taqsim.blocks_demo import plot_block_catalogue

_ = plot_block_catalogue()

Для каждого блока верны три утверждения:

- **Блок только перемещает воду.** Он принимает, удерживает, теряет или передаёт её дальше. Сам блок не анализирует всю сеть.
- **Правило блока — это карточка с инструкцией.** Несколько строк Python говорят: *зная, сколько воды пришло в этом месяце и сколько осталось в запасе, сделай следующее*. Каждое правило в этом блокноте можно прочитать меньше чем за минуту.
- **Блок ведёт журнал.** Каждый месяц он записывает, сколько воды получил, накопил, потерял, выпустил или недопоставил. Эти записи называются *событиями*. Чтобы понять результат, мы читаем журналы.

Любая модель TaqSim строится за четыре шага: создать пустую систему, добавить блоки, добавить связи, проверить и запустить. Время идёт по шагам; здесь один шаг равен месяцу. Год начинается в октябре, как гидрологический год Зарафшана. Объёмы выражены в миллионах кубических метров (Mm³).

## Как пользоваться блокнотом {#sec-how}

Код в этом блокноте свёрнут. У каждой ячейки есть заголовок, объясняющий её назначение, а перед ней — краткое объяснение обычными словами. Поэтому блокнот можно читать как руководство, не раскрывая код. Чтобы посмотреть, как выполнен расчёт, раскройте ячейку («Показать код» на веб-странице или треугольник слева в Colab). Чтобы изменить расчёт, внесите правку и снова выполните эту ячейку и все следующие. Это учебный пример: чтобы начать заново, вернитесь к исходному блокноту и выполните настройку.

Выполняйте ячейки сверху вниз. В Colab команда **Runtime → Run all** («Среда выполнения → Выполнить все») запускает весь блокнот примерно за минуту.

## Настройка {#sec-setup}

Следующая ячейка импортирует семь типов блоков и связь из TaqSim, типы событий для чтения журналов и вспомогательные функции этого занятия, которые строят рисунки и содержат исходные значения. В конце задаётся река: Зарафшан у Раватходжи, по одному значению на месяц — медиана за 2010–2023 годы.

In [ ]:
# @title Выполнить расчёт / показать код
#| label: setup
from dataclasses import dataclass
from typing import ClassVar

import numpy as np
import pandas as pd

# The seven kinds of block, the edge, the system and the pieces a rule is made of
from taqsim import Demand, Edge, Frequency, PassThrough, Reach, Sink, Source, Splitter, Storage, Strategy, TimeSeries, Timestep, WaterSystem
from taqsim.node import EVAPORATION, SEEPAGE, NoReachLoss, NoRouting

# The diary entries we will read
from taqsim.node import DeficitRecorded, WaterConsumed, WaterGenerated, WaterLost, WaterOutput, WaterPassedThrough, WaterReceived, WaterReleased, WaterSpilled, WaterStored

# This notebook's helpers (src/zarafshan_taqsim/blocks_demo.py): the numbers, the rules, the figures
from zarafshan_taqsim.blocks_demo import (
    CAPACITY_MM3,
    DEAD_STORAGE_MM3,
    FARM_NEED_MM3,
    INFLOW_MM3,
    MONTHS,
    START_STORAGE_MM3,
    CanalShare,
    Evaporation,
    Knobs,
    LagRouting,
    ReleaseTheNeed,
    SeepageLoss,
    balance_table,
    build_demo,
    monthly,
    plot_balance,
    plot_demo_network,
    plot_months,
    plot_where_the_water_went,
    rule_card,
    simulate_demo,
    system_balance,
)

river = np.array(INFLOW_MM3)  # the Zarafshan at Ravatkhoja, median month of 2010-2023, October first
print(f"The river brings {river.sum():,.0f} Mm³ a year: {river.min():.0f} in its lowest month, {river.max():.0f} in its highest.")

# Source и Sink: родник и море {#sec-source-sink}

Это два конца любой цепочки. **Source** (источник) каждый месяц подаёт в сеть объём воды из заданного списка. **Sink** (выходной узел) принимает воду на выходе из модели и записывает её количество. Отдельное правило не требуется: правило источника — его список значений, а выходному узлу ничего выбирать не нужно.

Самая маленькая система состоит из одного Source, одного Sink и одной связи. Ячейка ниже строит её за четыре обычных шага, затем читает два журнала: сколько воды река подала каждый месяц и сколько дошло до моря. Последняя строка проверяет совпадение этих объёмов в каждом месяце.

In [ ]:
# @title Выполнить расчёт / показать код
#| label: source-sink
system = WaterSystem(frequency=Frequency.MONTHLY)  # 1. an empty system; one step is one month
system.add_node(Source(id="river", inflow=TimeSeries(values=list(river))))  # 2. the blocks
system.add_node(Sink(id="sea"))
system.add_edge(Edge(id="river_to_sea", source="river", target="sea"))  # 3. the edge, named after its two ends
system.validate()  # 4. check that the network is complete, then run twelve months
system.simulate(12)

generated = monthly(system.nodes["river"], WaterGenerated, steps=12)  # the river's diary: what it put in
arrived = monthly(system.nodes["sea"], WaterReceived, steps=12)  # the sea's diary: what reached it
print("Every month the sea received exactly what the river generated:", np.allclose(generated, arrived))

In [ ]:
# @title Выполнить расчёт / показать код
#| label: fig-source
#| fig-cap: "The Source puts in what its list says: the Zarafshan's median year, low in winter, a melt peak in July."
#| echo: false
_ = plot_months({"the river": generated}, "A Source: one number per month, from a list")

Если раскрыть ячейку, сверху видно: `WaterSystem(frequency=Frequency.MONTHLY)` создаёт пустую систему с месячным шагом; `add_node` добавляет блок, `add_edge` — связь, названную по её двум концам; `validate()` проверяет, что из каждого блока можно дойти до выходного узла и что все адресаты правил существуют; `simulate(12)` моделирует двенадцать месяцев. После расчёта функция `monthly(block, EntryType, steps=12)` открывает журнал блока и суммирует события одного типа за каждый месяц. Это короткая вспомогательная функция; она встречается под рисунками и позволяет читать результаты TaqSim.

Ещё две детали. Имя связи `river_to_sea` появляется в журнале моря как `source_id` каждой подачи: так блок различает несколько входящих потоков. Источник создаёт столько воды, сколько указано в его списке, даже если ниже по течению никто не может её принять. Список — его единственное правило.

# Reach: коридор {#sec-reach}

**Reach** — участок реки или канала. Вода выходит из него позже, чем поступила, и часть может теряться по пути. У него две карточки: *модель перемещения потока* задаёт задержку, а *правило потерь* — объём потерь. В нашем учебном примере вся вода задерживается на один месяц, а десятая часть потока фильтруется в грунт.

In [ ]:
# @title Выполнить расчёт / показать код
#| label: reach-cards
#| echo: false
rule_card(LagRouting)

In [ ]:
# @title Выполнить расчёт / показать код
#| label: reach-loss-card
#| echo: false
rule_card(SeepageLoss)

Читаем карточку перемещения потока: `initial_state` — запас до первого месяца (здесь нулевой); `route` каждый месяц получает приток и возвращает текущий выпуск и оставшийся запас; `storage` сообщает объём внутри участка. Карточка потерь возвращает словарь с потерями по причинам, чтобы журнал показывал, *почему* вода потеряна. Ни одна из этих карточек не наследует `Strategy`: в этом примере оптимизатор не должен настраивать физические процессы.

Следующая ячейка помещает каньон между рекой и морем: добавляются один блок и одна связь. При создании каньону передают обе карточки (`routing_model=` и `loss_rule=`). После расчёта читаются три ряда из его журнала: месячный приток, потери и выпуск с другого конца.

In [ ]:
# @title Выполнить расчёт / показать код
#| label: reach
system = WaterSystem(frequency=Frequency.MONTHLY)
system.add_node(Source(id="river", inflow=TimeSeries(values=list(river))))
system.add_node(Reach(id="canyon", routing_model=LagRouting(months=1), loss_rule=SeepageLoss(share=0.1)))
system.add_node(Sink(id="sea"))
system.add_edge(Edge(id="river_to_canyon", source="river", target="canyon"))
system.add_edge(Edge(id="canyon_to_sea", source="canyon", target="sea"))
system.validate()
system.simulate(12)

canyon = system.nodes["canyon"]
entered = monthly(canyon, WaterReceived, steps=12)
leaked = monthly(canyon, WaterLost, steps=12)
came_out = monthly(canyon, WaterOutput, steps=12)

In [ ]:
# @title Выполнить расчёт / показать код
#| label: fig-reach
#| fig-cap: "What enters the canyon comes out one month later and a tenth smaller. In October nothing comes out yet; the first month's water is still inside."
#| echo: false
_ = plot_months({"entered the canyon": entered, "came out": came_out, "leaked away": leaked}, "A Reach: delay and loss")

Как читать рисунок: синяя линия — приток, оранжевая — выпуск, зелёная — потери. Оранжевая линия повторяет синюю со сдвигом на месяц вправо и уменьшением на одну десятую. Июльский пик 993 Mm³ достигает конца участка в августе в объёме 894 Mm³. В октябре выпуска нет: в начале года участок был пуст, а октябрьская вода ещё находится внутри. К концу года участок удерживает месячный объём, поэтому море получает меньше, чем подала река; разница остаётся в каньоне. В модели Зарафшана этого курса 35 таких участков — по одному на каждый отрезок реки или канала между двумя другими блоками.

# Splitter: развилка с регулировщиком {#sec-splitter}

**Splitter** (распределитель) направляет воду двум или нескольким получателям. Его карточка — *правило распределения* — указывает объём для каждого получателя по его имени. Наше правило отправляет фиксированную долю в канал, а остаток — дальше по реке.

In [ ]:
# @title Выполнить расчёт / показать код
#| label: split-card
#| echo: false
rule_card(CanalShare)

Эта карточка уже является `Strategy`: `__params__` задаёт параметр, который может менять оптимизатор (`share`), а `__bounds__` — допустимый диапазон. Ключи возвращаемого словаря — имена получателей, поэтому они должны совпадать с именами блоков, к которым ведут связи распределителя.

В следующей ячейке река поступает в головное водозаборное сооружение, направляющее 30 % воды в канал. Два выходных узла обозначают канал и речную ветвь: в их журналах видно, сколько получила каждая ветвь. Короткий цикл добавляет сразу три связи.

In [ ]:
# @title Выполнить расчёт / показать код
#| label: splitter
system = WaterSystem(frequency=Frequency.MONTHLY)
system.add_node(Source(id="river", inflow=TimeSeries(values=list(river))))
system.add_node(Splitter(id="headworks", split_policy=CanalShare(share=0.3)))
system.add_node(Sink(id="canal"))  # for now the two branches end right here
system.add_node(Sink(id="turbine"))
for a, b in (("river", "headworks"), ("headworks", "canal"), ("headworks", "turbine")):
    system.add_edge(Edge(id=f"{a}_to_{b}", source=a, target=b))
system.validate()
system.simulate(12)

to_canal = monthly(system.nodes["canal"], WaterReceived, steps=12)
to_river = monthly(system.nodes["turbine"], WaterReceived, steps=12)

In [ ]:
# @title Выполнить расчёт / показать код
#| label: fig-splitter
#| fig-cap: "Thirty percent into the canal, seventy down the river, every month. The two bars always add up to the river."
#| echo: false
_ = plot_months({"into the canal": to_canal, "down the river": to_river}, "A Splitter: so much this way, the rest that way", kind="bar", stacked=True)

На рисунке объёмы двух ветвей сложены: нижняя часть столбца — 30 % для канала, верхняя — 70 % для реки. Вместе они каждый месяц точно равны притоку. Правило распределения может зависеть от месяца (`t.index` — номер шага), поступившего объёма или информации, доступной блоку. Правило, которое летом направляет больше воды, чем зимой, потребует лишь нескольких дополнительных строк; вы напишете его в упражнении 1.

# Storage: ванна с выпускным клапаном {#sec-storage}

**Storage** — водохранилище. Представьте ванну: вода поступает и заполняет её, при переполнении избыток переливается, часть испаряется, а клапан внизу выпускает заданный объём. Ниже клапана остаётся вода, которую нельзя выпустить (*мёртвый объём*). У блока две карточки: *правило выпуска* для клапана и *правило потерь* для испарения.

In [ ]:
# @title Выполнить расчёт / показать код
#| label: release-card
#| echo: false
rule_card(ReleaseTheNeed)

In [ ]:
# @title Выполнить расчёт / показать код
#| label: evaporation-card
#| echo: false
rule_card(Evaporation)

Правило выпуска читает месячную потребность хозяйства и пытается выпустить соответствующий объём. При этом выпуск не превышает пропускную способность (`cap`) и доступный запас выше мёртвого объёма. Правило потерь забирает три процента воды в водохранилище.

На каждом шаге порядок фиксирован: **принять воду, пополнить запас (сбросить избыток при переполнении), учесть потери, выполнить выпуск**. Это важно: испарение рассчитывается уже с учётом притока текущего месяца, а выпуск — из оставшегося после потерь запаса.

Следующая ячейка подаёт в водохранилище пятую часть речного притока, как будто головное сооружение направляет эту долю в канал. Расчёт длится два года, чтобы второй год отражал установившийся сезонный режим. У водохранилища пять настроек: полная ёмкость (`capacity`), недоступный для выпуска запас (`dead_storage`), начальный запас (`initial_storage`), правило выпуска (`release_policy`) и правило потерь (`loss_rule`). После расчёта читаются четыре ряда журнала — приток, сброс при переполнении, испарение и регулируемый выпуск; для рисунков берётся второй год.

Запас воды не является отдельным событием журнала. Его получают из начального запаса, последовательно прибавляя приток и вычитая все расходы за каждый месяц. Ячейка выполняет такое суммирование и в последней строке сверяет результат с запасом самого блока.

In [ ]:
# @title Выполнить расчёт / показать код
#| label: storage
canal_water = river * 0.2  # the canal's share, as if the headworks sent a fifth of the river into the tub
system = WaterSystem(frequency=Frequency.MONTHLY)
system.add_node(Source(id="canal", inflow=TimeSeries(values=list(canal_water) * 2)))  # two years: the second is shown
system.add_node(
    Storage(
        id="reservoir",
        capacity=CAPACITY_MM3,  # 667 Mm³, the Kattakurgan reservoir
        dead_storage=DEAD_STORAGE_MM3,  # 66.7 Mm³ the valve cannot reach
        initial_storage=START_STORAGE_MM3,  # 170 Mm³ on the first of October
        release_policy=ReleaseTheNeed(cap=60.0),
        loss_rule=Evaporation(share=0.03),
    )
)
system.add_node(Sink(id="farm"))
system.add_edge(Edge(id="canal_to_reservoir", source="canal", target="reservoir"))
system.add_edge(Edge(id="reservoir_to_farm", source="reservoir", target="farm"))
system.validate()
system.simulate(24)

tub = system.nodes["reservoir"]
second_year = slice(12, 24)
stored = monthly(tub, WaterStored, steps=24)  # what went into the tub each month (not the level)
overflowed = monthly(tub, WaterSpilled, steps=24)
evaporated = monthly(tub, WaterLost, steps=24)
released = monthly(tub, WaterReleased, steps=24)
level = START_STORAGE_MM3 + np.cumsum(stored - evaporated - released)  # the level at the end of each month
stored, overflowed, evaporated, released, level = (x[second_year] for x in (stored, overflowed, evaporated, released, level))
print(f"The tub ends the second year holding {tub.storage:,.0f} Mm³; the level path says {level[-1]:,.0f}.")

In [ ]:
# @title Выполнить расчёт / показать код
#| label: fig-storage
#| fig-cap: "The second year of the tub. From May to August the melt does not fit and the rest overflows; the valve lets out the farm's need, at most 60; the vapour takes three percent a month, about 19 Mm³. The dashed line is the need."
#| echo: false
_ = plot_months(
    {"went into the tub": stored, "overflowed": overflowed, "evaporated": evaporated, "released by the valve": released},
    "A Storage: receive, store, lose, release",
    kind="bar",
    reference=("the farm's need", np.array(FARM_NEED_MM3)),
)

In [ ]:
# @title Выполнить расчёт / показать код
#| label: fig-level
#| fig-cap: "The level at the end of each month of the second year, against the rim (667) and the dead storage (66.7). The level never shows 667 at month end: within the month the valve and the vapour already took their share."
#| echo: false
fig = plot_months({"water in the tub": level}, "How full the tub is", unit="Mm³")
ax = fig.axes[0]
ax.axhline(CAPACITY_MM3, color="#6b6a66", lw=1, ls="--")
ax.axhline(DEAD_STORAGE_MM3, color="#6b6a66", lw=1, ls=":")
ax.set_ylim(0, 720)

Как читать два рисунка: на первом показаны месячные приток (синий), сброс при переполнении (оранжевый), испарение (зелёный) и регулируемый выпуск (жёлтый), а пунктиром — потребность хозяйства. На втором — запас к концу месяца между двумя серыми линиями: полной ёмкостью 667 и мёртвым объёмом 66.7 Mm³. Выпуск точно следует потребности до июля, когда потребность 83 Mm³ превышает предел 60 Mm³: жёлтый столбец не достигает пунктира.

Запомните: в TaqSim **сброс при переполнении полного водохранилища идёт дальше по выходящей связи**, здесь — к хозяйству, как и регулируемый выпуск. Он записывается как `WaterSpilled`, чтобы их можно было различать. Далее мы встретим другой тип сброса — у блока с ограниченной пропускной способностью.

# Demand: потребитель {#sec-demand}

**Demand** — потребитель воды: хозяйство, город или электростанция. У него три параметра. *Потребность* — запрашиваемый месячный объём из списка. *Доля безвозвратного потребления* — часть полученной воды, которая расходуется; остаток возвращается ниже по течению как дренажная или охлаждающая вода. *Коэффициент эффективности доставки* — доля воды на входе, доходящая до места использования; остальное теряется по пути от канала к корням растений.

Если потребитель получает меньше необходимого, он записывает дефицит в событии `DeficitRecorded`. На этих записях строятся показатели дефицита в данном курсе.

Следующая ячейка подаёт воду из канала прямо хозяйству, без водохранилища; неиспользованная вода поступает в дренаж. Задаются месячные потребности (`requirement`), доля безвозвратного потребления (`consumption_fraction`) и эффективность доставки (`efficiency`). После расчёта читаются пять рядов: поступление, потери доставки, потребление, возврат и дефицит. Проверяется, что поступление равно сумме потерь, потребления и возврата: водный баланс должен сходиться каждый месяц.

In [ ]:
# @title Выполнить расчёт / показать код
#| label: demand
system = WaterSystem(frequency=Frequency.MONTHLY)
system.add_node(Source(id="canal", inflow=TimeSeries(values=list(canal_water))))
system.add_node(
    Demand(
        id="farm",
        requirement=TimeSeries(values=list(FARM_NEED_MM3)),  # what the crops need each month
        consumption_fraction=0.7,  # the crops use up 70 %; 30 % drains back
        efficiency=0.85,  # 85 % of the water at the gate reaches the roots
    )
)
system.add_node(Sink(id="drain"))
system.add_edge(Edge(id="canal_to_farm", source="canal", target="farm"))
system.add_edge(Edge(id="farm_to_drain", source="farm", target="drain"))
system.validate()
system.simulate(12)

farm = system.nodes["farm"]
received = monthly(farm, WaterReceived, steps=12)
lost_on_the_way = monthly(farm, WaterLost, steps=12)
used_up = monthly(farm, WaterConsumed, steps=12)
drained_back = monthly(farm, WaterOutput, steps=12)
short = monthly(farm, DeficitRecorded, steps=12, field="deficit")
print("The farm's books close every month:", np.allclose(received, lost_on_the_way + used_up + drained_back))

In [ ]:
# @title Выполнить расчёт / показать код
#| label: fig-demand
#| fig-cap: "The canal brings more than the farm needs in most months, so the farm uses what it needs, loses 15 % on the way, and sends the rest back. In September the canal brings 70 for a need of 67, only 60 reach the roots, and the farm is short."
#| echo: false
_ = plot_months(
    {"used up by the crops": used_up, "lost on the way": lost_on_the_way, "drained back": drained_back, "short": short},
    "A Demand: need, use, loss, return",
    kind="bar",
    stacked=True,
    reference=("the farm's need", np.array(FARM_NEED_MM3)),
)

Как читать рисунок: части столбцов показывают, куда пошла вода хозяйства — потребление, потери доставки и возвратный сток (синий, оранжевый и зелёный). Сверху жёлтым показан дефицит; ступенчатый пунктир обозначает потребность. Начните с августа: канал доставил 142 при потребности 113 Mm³; до корней теряется 15 %, доходит 121 Mm³ — этого достаточно. Теперь рассмотрим сентябрь:

In [ ]:
# @title Выполнить расчёт / показать код
#| label: demand-table
#| echo: false
pd.DataFrame({"month": MONTHS, "need": FARM_NEED_MM3, "received": received.round(1), "reached the roots": (received - lost_on_the_way).round(1), "short": short.round(1)}).style.hide(axis="index")

Сентябрьская строка показывает главное: канал доставляет 70, до корней доходит 60, а потребность равна 67 Mm³. **Потери доставки возникают между входом и местом потребления.** Если верхний блок подаст ровно потребность, хозяйству не хватит объёма, потерянного по пути. Учитывайте это при настройке выпуска водохранилища.

# PassThrough: ворота ограниченной ширины {#sec-passthrough}

**PassThrough** — простой пропускной блок, похожий на ворота. Он пропускает воду в пределах заданной *пропускной способности*, а остаток записывает как сброс. Так можно представить турбину, насос, трубу или водозабор — любой объект с верхним пределом расхода. Без ограничения он служит точкой учёта.

В следующей ячейке турбина находится между рекой и морем. Её единственная настройка — предел 600 Mm³ в месяц. После расчёта читаются два ряда журнала: пропущенная вода и сброс сверх ограничения. Затем выводится доля речной воды, достигшая моря.

In [ ]:
# @title Выполнить расчёт / показать код
#| label: passthrough
system = WaterSystem(frequency=Frequency.MONTHLY)
system.add_node(Source(id="river", inflow=TimeSeries(values=list(river))))
system.add_node(PassThrough(id="turbine", capacity=600.0))  # the turbine can take 600 Mm³ a month
system.add_node(Sink(id="sea"))
system.add_edge(Edge(id="river_to_turbine", source="river", target="turbine"))
system.add_edge(Edge(id="turbine_to_sea", source="turbine", target="sea"))
system.validate()
system.simulate(12)

turbine = system.nodes["turbine"]
through = monthly(turbine, WaterPassedThrough, steps=12)
past = monthly(turbine, WaterSpilled, steps=12)
print(f"The sea received {monthly(system.nodes['sea'], WaterReceived, steps=12).sum():,.0f} of the river's {river.sum():,.0f} Mm³; {past.sum():,.0f} spilled past the gate.")

In [ ]:
# @title Выполнить расчёт / показать код
#| label: fig-passthrough
#| fig-cap: "June to August the river is wider than the gate. What does not fit is recorded as spilled and leaves the model."
#| echo: false
_ = plot_months({"through the gate": through, "spilled past it": past}, "A PassThrough: what fits goes through", kind="bar", stacked=True)

Это второй тип сброса, отличающийся от перелива водохранилища: **вода сверх пропускной способности PassThrough покидает учитываемую модельную систему.** TaqSim регистрирует её, но дальше не отслеживает. В реальной реке обходящая турбину вода остаётся в русле. Если это важно для задачи, турбину следует поставить на отдельную ветвь после Splitter либо ограничение задать на Reach. Для правильного чтения результата нужно различать эти два смысла сброса.

# Все девять блоков вместе {#sec-together}

Теперь соберём всю цепочку: река входит в каньон, затем в головное сооружение; оно направляет часть воды в канал, а остальную — к турбине на реке. Канал питает водохранилище, водохранилище снабжает хозяйство, обе ветви заканчиваются выходными узлами. Всего девять блоков семи типов, восемь связей и пять карточек с настраиваемыми параметрами.

In [ ]:
# @title Выполнить расчёт / показать код
#| label: fig-network
#| fig-cap: "The demo system. Water moves left to right; the kind of each block is written under its name."
#| echo: false
_ = plot_demo_network()

На этот раз связи собирает вспомогательная функция `build_demo`, чтобы одну и ту же систему можно было многократно создавать с разными настройками. Пять доступных участнику параметров собраны в небольшую структуру `Knobs`. Ниже показаны две карточки: сама структура и построение сети — те же четыре шага, которые вы уже видели для отдельных блоков.

In [ ]:
# @title Выполнить расчёт / показать код
#| label: build-card
#| echo: false
rule_card(Knobs)

In [ ]:
# @title Выполнить расчёт / показать код
#| label: build-demo-card
#| echo: false
rule_card(build_demo)

Функция `simulate_demo` строит систему с заданными настройками, моделирует три года и возвращает последний, когда водохранилище уже вошло в сезонный режим. Журналы всех блоков преобразованы в месячные ряды. Следующая ячейка запускает настройки по умолчанию, а рисунок показывает июль — месяц максимального речного притока — с объёмами воды на каждой связи.

In [ ]:
# @title Выполнить расчёт / показать код
#| label: run
run = simulate_demo(Knobs())  # three years; the last one comes back, month by month, block by block

In [ ]:
# @title Выполнить расчёт / показать код
#| label: fig-july
#| fig-cap: "July, the river's biggest month. Follow the numbers along the edges: 993 enters the canyon, the 771 of June comes out, 154 of it goes into the canal and 617 to the turbine, which can only take 600."
#| echo: false
_ = plot_demo_network(run, month=9)

In [ ]:
# @title Выполнить расчёт / показать код
#| label: fig-where
#| fig-cap: "Where the water went, month by month. Most of it goes to the river end through the turbine; the canyon holds each month's water for a month (the light bars below zero are last month's water coming out); the reservoir fills in spring and drains in late summer."
#| echo: false
_ = plot_where_the_water_went(run)

Как читать распределение воды: каждый столбец соответствует месяцу, а его части — конечному распределению поступившей воды. В сумме они дают пунктирную линию притока. Тёмно-синяя часть снизу — вода, дошедшая до конца речной ветви через турбину; голубая выше — сброс сверх пропускной способности турбины в многоводные месяцы. Бледно-голубая часть — запас каньона для следующего месяца; та же часть ниже нуля — выход воды прошлого месяца. Тёмная часть ближе к верху — накопление в водохранилище, а ниже нуля — расходование его запаса в конце лета. Тонкие жёлтые и оранжевые части — потери; зелёные — потребление и возвратный сток хозяйства.

## Каждая капля учтена {#sec-balance}

Журнал каждого блока позволяет проверить: поступление равно сумме выхода, потребления, потерь, воды, покинувшей модель через ограничение PassThrough, и изменения запаса. Таблица проверяет это за год. Столбцы: `in` — поступление; `out` — передача следующему блоку; `consumed` — безвозвратное потребление посевами; `lost` — фильтрация, испарение и потери в хозяйстве; `left` — вода, покинувшая модель через PassThrough; `stored` — годовое изменение запаса; `closes` — поступление минус все остальные члены баланса. Последнее должно равняться нулю.

In [ ]:
# @title Выполнить расчёт / показать код
#| label: balance-table
#| echo: false
balance_table(run).round(1).style.hide(axis="index").format(precision=1)

Во всех строках `closes` равен нулю. В `out` водохранилища включён сброс при переполнении, потому что он дошёл до хозяйства. Значение `left` турбины — вода, которую TaqSim больше не отслеживает. Теперь покажем баланс всей системы на одном рисунке:

In [ ]:
# @title Выполнить расчёт / показать код
#| label: fig-balance
#| fig-cap: "The year adds up. What the river brought equals what reached the two sinks, what the crops used, what was lost, what left at the turbine's gate, and the change in storage (zero: the reservoir ended the year where it began)."
#| echo: false
_ = plot_balance(run)

In [ ]:
# @title Выполнить расчёт / показать код
#| label: balance-numbers
#| echo: false
b = system_balance(run)
stored = 0.0 if abs(b["stored"]) < 0.5 else b["stored"]
print(f"River {b['river']:,.0f} = sinks {b['to_sinks']:,.0f} + used {b['consumed']:,.0f} + lost {b['lost']:,.0f} + left at the gate {b['left']:,.0f} + stored {stored:,.0f}")

Эту проверку стоит сделать привычкой. Если баланс отдельных блоков не сходится, в построении сети есть ошибка. Сходящийся баланс ещё не доказывает правильность модели, но хотя бы позволяет указать, где находится каждая капля.

## Изменим параметр {#sec-knobs}

Проведём два опыта. `Knobs(release_cap=20.0)` задаёт предел выпуска 20, оставляя остальные четыре параметра по умолчанию. Сначала ограничим выпуск водохранилища до 20 Mm³ в месяц. Ячейка запускает модель и выводит годовой дефицит хозяйства рядом с результатом при обычном пределе 60 Mm³.

In [ ]:
# @title Выполнить расчёт / показать код
#| label: tight-valve
tight = simulate_demo(Knobs(release_cap=20.0))
print(f"Valve at 20: the farm is short {tight.series['farm']['deficit'].sum():,.0f} Mm³ in the year, while the reservoir never drops below {tight.storage_level.min():,.0f} Mm³.")
print(f"Valve at 60: the farm is short {run.series['farm']['deficit'].sum():,.0f} Mm³.")

In [ ]:
# @title Выполнить расчёт / показать код
#| label: fig-tight
#| fig-cap: "A full reservoir and a thirsty farm: with the valve at 20 the farm is short in May and June while the tub holds over 600 Mm³. The shortfall is the rule's doing, not the river's."
#| echo: false
_ = plot_months(
    {"short, valve at 20": tight.series["farm"]["deficit"], "short, valve at 60": run.series["farm"]["deficit"]},
    "The same river, two valves",
    kind="bar",
    reference=("the farm's need", np.array(FARM_NEED_MM3)),
)

Во втором опыте уменьшим пропускную способность турбины с 600 до 300 Mm³ в месяц. Ячейка покажет, сколько воды сбрасывается сверх этого предела и покидает модель и насколько меньше воды доходит до конца речной ветви.

In [ ]:
# @title Выполнить расчёт / показать код
#| label: narrow-gate
narrow = simulate_demo(Knobs(turbine_capacity=300.0))
print(f"Gate at 300: {narrow.series['turbine']['spilled'].sum():,.0f} Mm³ spill past the turbine and leave the model, against {run.series['turbine']['spilled'].sum():,.0f} at 600.")
print(f"The river end receives {narrow.flows['turbine_to_river_end'].sum():,.0f} instead of {run.flows['turbine_to_river_end'].sum():,.0f} Mm³.")

Почему более строгий предел выпуска не даёт ещё большего дефицита? Посмотрите на месяцы: хозяйству не хватает воды в мае и июне, когда регулируемый выпуск — единственный выход из водохранилища. С июля водохранилище полно, талые воды переливаются и доходят до хозяйства независимо от правила выпуска. В одном опыте видны два вывода: правило управления может вызывать дефицит, а сброс при переполнении водохранилища продолжает движение по сети.

Измените любой из пяти параметров `Knobs(...)`, повторите расчёт и прочитайте журналы. Игра этого шага делает то же самое с помощью ползунков: результаты всех комбинаций рассчитаны заранее.

## На что обратить внимание {#sec-notice}

1. **Каньон задерживает воду на месяц.** Июльский пик достигает головного сооружения в августе. Задержка в Reach меняет время поступления, а потери — объём.
2. **Два типа сброса имеют разный смысл.** Перелив водохранилища идёт дальше по связи. Сброс сверх ограничения PassThrough покидает модель. Оба записаны как `WaterSpilled`; смысл определяется типом блока.
3. **Причиной дефицита может быть правило.** При пределе выпуска 20 Mm³ хозяйство испытывает дефицит в мае и июне, хотя в водохранилище более 600 Mm³. Прежде чем винить реку, проверьте правило.
4. **Потери доставки возникают до удовлетворения потребности.** Даже при широком выпуске остаётся небольшой дефицит: правило задаёт потребность на входе, а затем теряется 15 %. Учитывающее потери правило должно выпускать потребность, делённую на коэффициент эффективности.

# Попробуйте сами {#sec-try}

1. **Сезонное распределение.** Напишите правило `SummerShare`: с апреля по сентябрь направлять 40 % в канал (`t.index % 12` от 6 до 11), в остальные месяцы — 10 %. Примените его к головному сооружению и сравните динамику запаса водохранилища с исходной.
2. **Более разумный выпуск.** Измените `ReleaseTheNeed`, чтобы выпускать потребность, делённую на эффективность доставки хозяйства. Исчезает ли дефицит во всех месяцах? Если нет, то где и почему?
3. **Сезонное испарение.** Задайте в `Evaporation` 1 % зимой и 5 % летом. Как изменится годовая подача хозяйству?
4. **Второй потребитель.** Добавьте между водохранилищем и хозяйством город — блок `Demand` с потребностью 10 Mm³ в месяц и возвратом 80 % полученной воды. Кому первым не хватит воды в сухой год и как это связано с порядком блоков?

**С Gemini в Colab.** Нажмите синюю кнопку Gemini и вставьте один из запросов ниже. Для этого упражнения справочник помощника — код самого блокнота, поэтому укажите нужную ячейку:

> По образцу класса `CanalShare` в этом блокноте напиши правило, направляющее 40 % воды в канал в месяцах с 6 по 11 каждого года и 10 % в остальные месяцы. Затем собери демонстрационную систему с этим правилом и выведи минимальный запас водохранилища.

> Измени `ReleaseTheNeed`: выпускай `need / 0.85` вместо `need`, сохранив ограничение `cap`. Запусти `simulate_demo` с новым правилом и выведи годовой дефицит хозяйства до и после изменения.

# Куда двигаться дальше {#sec-next}

В стартовом блокноте есть краткая памятка по блокам и правилам TaqSim v0.1.4 для ИИ-помощника. Учебник «Водохранилище и хозяйство» строит систему из четырёх блоков и передаёт настройку параметров оптимизатору. Учебник моделирования рассматривает реальную сеть: 61 блок и шесть лет посуточного расчёта, построенные из тех же семи типов. Документация TaqSim также входит в пакет: `from taqsim import get_docs_path`.

# Источники {#sec-sources .unnumbered}

- Река: Зарафшан у Раватходжи, суточные наблюдения 2010–2023 годов, медианные месячные значения (`data/ZRB_baseline/inflow/`).
- Хозяйство: месячная потребность Каттакурганского района в орошении из модели Зарафшана этого курса.
- Водохранилище: Каттакурганское, полная ёмкость 667 Mm³, мёртвый объём 66.7 Mm³, начальный запас на 1 октября — 170 Mm³ (медиана за 2000–2022 годы; см. игру с шестнадцатью блоками).
- TaqSim v0.1.4, hydrosolutions GmbH, https://github.com/hydrosolutions/taqsim и документация по узлам, событиям и стратегиям.